# CS781 Phase 4 - final analysis, ensembles and submission files (CPU only)
Reads the three run bundles (`bundle_p4_1.zip`, `bundle_p4_2.zip`, `bundle_p4_3.zip`; zipped or already unzipped by Kaggle)
and builds every table for the report, the ensembles and the submission files. Nothing is trained on dev or test:
ensembles average saved probabilities; members and decision thresholds are chosen on dev (so dev scores of
ensembles / calibrated thresholds are optimistic - the leaderboard is the honest check).

1. All runs; Table 1 technique x model; Table 2 representation study; Table 3 cost; Table 4 class balance + calibration
2. Table 5 which parts of the judgment the attention uses; Table 6 bag-of-words baseline (how much the LM adds)
3. Table 7 ensembles; Table 8 error analysis (false positives / false negatives)
4. Submission files in submission order, each a different method (`submissions/NN_<name>.zip`, `SUBMISSIONS.csv`,
   `all_submissions.zip`)

**Inputs:** the three bundles + `cs781-p4-code`. Internet ON, `HF_TOKEN`, **Accelerator: None**. Save & Run All (~10-20 min,
mostly the bag-of-words baseline; set `RUN_BOW = False` to skip it).

In [1]:
import os, sys, json, re, glob, shutil, time, zipfile
import numpy as np
W = "/kaggle/working"
WORK, P4DATA, RUNS = f"{W}/p4code", "/tmp/p4data", "/tmp/p4runs"
for d in (RUNS, P4DATA):
    os.makedirs(d, exist_ok=True)
def find_dir(name, root="/kaggle/input"):
    hits = sorted(glob.glob(os.path.join(root, "**", name), recursive=True), key=len)
    assert hits, f"{name} not found under {root}"
    return os.path.dirname(hits[0])
shutil.rmtree(WORK, ignore_errors=True)
shutil.copytree(find_dir("cjpe_train.py"), WORK)
os.chdir(WORK); sys.path.insert(0, WORK)
import cjpe_data as D, cjpe_eval as E
from huggingface_hub import hf_hub_download, list_repo_files
from kaggle_secrets import UserSecretsClient
REPO = "Exploration-Lab/CS-781-Capstone"
tok_ = UserSecretsClient().get_secret("HF_TOKEN")
for f in [f for f in list_repo_files(REPO, repo_type="dataset", token=tok_) if f.startswith("Phase_4")]:
    dst = os.path.join(P4DATA, f); os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.copy(hf_hub_download(REPO, f, repo_type="dataset", token=tok_), dst)
EXPS = {e["tag"]: e for e in json.load(open(f"{WORK}/experiments_p4.json"))}
for z in glob.glob("/kaggle/input/**/bundle_p4_*.zip", recursive=True):
    zipfile.ZipFile(z).extractall(RUNS)
for p in glob.glob("/kaggle/input/**/summary.json", recursive=True):       # bundles Kaggle already unzipped
    d = os.path.dirname(p); t = os.path.basename(d)
    if t in EXPS and not os.path.exists(f"{RUNS}/{t}"):
        shutil.copytree(d, f"{RUNS}/{t}")
FOUND = sorted(t for t in os.listdir(RUNS) if t in EXPS and os.path.exists(f"{RUNS}/{t}/dev_prob.npz"))
print(len(FOUND), "runs found; missing:", " ".join(sorted(set(EXPS) - set(FOUND))) or "none")

Phase_4_CJPE/Test/test.json: reconstructing file:   0%|          |  0.00B / 35.3MB            

Phase_4_CJPE/Test/test.json: downloading bytes:           |  0.00B            

Phase_4_CJPE/Train/dev.json: reconstructing file:   0%|          |  0.00B / 45.0MB            

Phase_4_CJPE/Train/dev.json: downloading bytes:           |  0.00B            

Phase_4_CJPE/Train/train.json: reconstructing file:   0%|          |  0.00B /  737MB            

Phase_4_CJPE/Train/train.json: downloading bytes:           |  0.00B            

56 runs found; missing: none


## Tables 1-5

In [2]:
S = {t: json.load(open(f"{RUNS}/{t}/summary.json")) for t in FOUND}
def npz(t, split):
    p = f"{RUNS}/{t}/{split}_prob.npz"
    return dict(np.load(p)) if os.path.exists(p) else None
DEV = {t: npz(t, "dev") for t in FOUND}
TEST = {t: npz(t, "test") for t in FOUND}
ref_ids = [str(i) for i in DEV[FOUND[0]]["ids"]]
y_dev = DEV[FOUND[0]]["y"].astype(int)
for t in FOUND:
    assert [str(i) for i in DEV[t]["ids"]] == ref_ids, f"{t}: dev order differs"
HAVE_TEST = all(TEST[t] is not None for t in FOUND)
test_ids = [str(i) for i in TEST[FOUND[0]]["ids"]] if HAVE_TEST else []
dev_docs = {x["id"]: x for x in D.load_split(D.find_split(P4DATA, "dev"))}
info = S[FOUND[0]]["cache_info"]
f1 = lambda t: S[t]["dev"]["macro_f1"]
MODELS, TECHS = "ABCDE", [k for k in ("lorma", "adalora", "adapter", "bitfit", "gradual", "partial", "full", "lora")]
RUN = {(EXPS[t]["model"].upper(), t.split("_", 1)[1]): t for t in FOUND}
L = ["=== REPORT phase 4 ===",
     f"data: {json.dumps(info)}",
     f"{len(FOUND)} runs; dev {len(y_dev)} cases ({int(y_dev.sum())} label 1); test {len(test_ids)} cases", "",
     "ALL RUNS by dev macro-F1 (decision 0.5) | AUC | macro-F1 at the dev-best threshold | P | R | predicted-1 share | "
     "steps (best/planned) | trainable M | train min | peak GB | ms/doc"]
for t in sorted(FOUND, key=f1, reverse=True):
    s, d = S[t], S[t]["dev"]
    L.append(f"{t:26s} {d['macro_f1']:.4f} | {d['auc']:.4f} | {d['macro_f1_best_threshold']:.4f}@{d['best_threshold']:.2f} | "
             f"{d['macro_p']:.4f} | {d['macro_r']:.4f} | {100 * d['pred_1_share']:3.0f}% | {s['best_step']}/{s.get('steps_planned', 0)} | "
             f"{s['params_trainable'] / 1e6:7.2f} | {s['train_minutes']:5.1f} | {s.get('peak_mem_gb') or 0:4.1f} | {s['infer_ms_per_doc']:6.1f}")
cell = lambda v, fmt="{:8.4f}": fmt.format(v) if v is not None else f"{'-':>8s}"
def table(title, key):
    L.append(f"\n{title}")
    L.append(f"{'':22s}" + "".join(f"{m:>8s}" for m in MODELS) + f"{'mean':>8s}")
    for tk in TECHS:
        v = [key(RUN[(m, tk)]) if (m, tk) in RUN else None for m in MODELS]
        ok = [x for x in v if x is not None]
        if ok:
            L.append(f"{tk:22s}" + "".join(cell(x) for x in v) + cell(np.mean(ok)))
    v = [np.mean([key(RUN[(m, tk)]) for tk in TECHS if (m, tk) in RUN]) if any((m, tk) in RUN for tk in TECHS) else None for m in MODELS]
    L.append(f"{'mean over techniques':22s}" + "".join(cell(x) for x in v))
table("TABLE 1a - TECHNIQUE x MODEL: dev macro-F1 (decision 0.5)", f1)
table("TABLE 1b - TECHNIQUE x MODEL: dev ROC-AUC (threshold-free)", lambda t: S[t]["dev"]["auc"])
table("TABLE 1c - TECHNIQUE x MODEL: dev macro-F1 at the dev-best threshold (optimistic)", lambda t: S[t]["dev"]["macro_f1_best_threshold"])

L.append("\nTABLE 2 - REPRESENTATION STUDY (LoRMA, same 8,000-case training subset): dev macro-F1 | AUC | change vs reference")
for m in MODELS:
    reps = sorted(t for t in FOUND if EXPS[t]["group"] == "R" and EXPS[t]["model"].upper() == m)
    if not reps:
        continue
    ref = next((t for t in reps if t.endswith("rep_ref")), None)
    L.append(f"model {m}:")
    for t in reps:
        d = S[t]["dev"]
        dd = (d["macro_f1"] - f1(ref), d["auc"] - S[ref]["dev"]["auc"]) if ref else (0, 0)
        L.append(f"{t:22s} {EXPS[t]['change'][:42]:42s} {d['macro_f1']:.4f} | {d['auc']:.4f} | {dd[0]:+.4f} | {dd[1]:+.4f} | "
                 f"{S[t]['train_minutes']:5.1f} min | {S[t]['infer_ms_per_doc']:6.1f} ms/doc")

L.append("\nTABLE 3 - COST per technique (mean over models): trainable params M | % of model | peak train GB | train min | "
         "train tok/s | inference ms/doc | saved model MB")
for tk in TECHS:
    ts = [RUN[(m, tk)] for m in MODELS if (m, tk) in RUN]
    if ts:
        g = lambda k: np.mean([S[t].get(k) or 0 for t in ts])
        L.append(f"{tk:10s} {g('params_trainable') / 1e6:7.2f}M | {100 * np.mean([S[t]['params_trainable'] / S[t]['params_total'] for t in ts]):5.1f}% | "
                 f"{g('peak_mem_gb'):4.1f} | {g('train_minutes'):5.1f} | {g('tok_s'):7,.0f} | {g('infer_ms_per_doc'):6.1f} | {g('model_mb'):7.1f}")
L.append("per model (LoRMA run): total params M | inference ms/doc | peak inference GB")
for m in MODELS:
    if (m, "lorma") in RUN:
        s = S[RUN[(m, "lorma")]]
        L.append(f"  {m}: {s['params_total'] / 1e6:7.2f}M | {s['infer_ms_per_doc']:6.1f} | {s.get('infer_mem_gb') or 0:4.2f}")

L.append(f"\nTABLE 4 - CLASS BALANCE: train label 1 = {info['train']['label_1']}, label 0 = {info['train']['label_0']}; "
         f"dev label 1 = {int(y_dev.sum())}/{len(y_dev)}")
L.append("predicted-1 share on dev at 0.5 (ideal ~ dev share) and the dev-best threshold, per run:")
for t in sorted(FOUND, key=f1, reverse=True)[:12]:
    d = S[t]["dev"]
    L.append(f"  {t:26s} pred-1 {100 * d['pred_1_share']:4.0f}%  class-0 F1 {d['per_class']['0']['f1']:.3f}  class-1 F1 "
             f"{d['per_class']['1']['f1']:.3f}  thr {d['best_threshold']:.2f} ({d['macro_f1']:.4f} -> {d['macro_f1_best_threshold']:.4f})")

# ---- Table 5: where the attention goes (relative position of the chunk in the judgment)
L.append("\nTABLE 5 - WHICH PARTS OF THE JUDGMENT THE MODEL USES: mean attention weight by relative chunk position (dev)")
bins = np.linspace(0, 1, 6)
for t in [x for x in FOUND if x.endswith(("rep_spread8", "rep_ref", "rep_firstlast"))] + [RUN[(m, "lorma")] for m in MODELS if (m, "lorma") in RUN][:2]:
    a, p = DEV[t]["att"], DEV[t]["pos"]
    ok = np.isfinite(a) & np.isfinite(p)
    if not ok.any():
        continue
    k = np.clip(np.digitize(p[ok], bins) - 1, 0, 4)
    share = [a[ok][k == b].sum() / a[ok].sum() for b in range(5)]
    L.append(f"  {t:26s} " + "  ".join(f"{int(100 * bins[b])}-{int(100 * bins[b + 1])}%: {100 * share[b]:4.1f}" for b in range(5))
             + f"   (uniform over the chunks read would give each bin its share of chunks)")
    if t.endswith("rep_spread8"):
        nb = np.bincount(k, minlength=5) / ok.sum()
        L.append(f"  {'(share of chunks per bin)':26s} " + "  ".join(f"{100 * x:4.1f}" for x in nb))

## Table 6: bag-of-words baseline

In [3]:
# ---- Table 6: bag-of-words baseline (no language model) - how much does the Phase-1 LM add?
RUN_BOW = False                         # True adds a TF-IDF baseline (slow on 32k cases, memory-heavy)
if RUN_BOW:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression
    t0 = time.time()
    trs = D.load_split(D.find_split(P4DATA, "train"))
    tp_ = D.find_split(P4DATA, "test")
    others = list(dev_docs.values()) + (D.load_split(tp_) if tp_ else [])
    held, held_h = {x["id"] for x in others}, {D.text_hash(x["text"]) for x in others}
    trs = [x for x in trs if x["id"] not in held and D.text_hash(x["text"]) not in held_h]   # as in the cache
    dv = [dev_docs[i] for i in ref_ids]
    last = lambda x, n: " ".join(D.clean_text(x["text"]).split()[-n:])
    for name, fn in (("last 800 words", lambda x: last(x, 800)), ("whole document", lambda x: D.clean_text(x["text"]))):
        vec = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=300000, sublinear_tf=True, dtype=np.float32)
        Xtr = vec.fit_transform([fn(x) for x in trs]); Xdv = vec.transform([fn(x) for x in dv])
        clf = LogisticRegression(C=4.0, max_iter=2000, class_weight="balanced").fit(Xtr, [x["label"] for x in trs])
        p = clf.predict_proba(Xdv)[:, 1]
        r = E.binary_report(y_dev, (p >= 0.5).astype(int))
        L.append(f"\nTABLE 6 - BAG-OF-WORDS BASELINE (TF-IDF 1-2 grams + logistic regression, {name}): dev macro-F1 "
                 f"{r['macro_f1']:.4f}, AUC {E.auc(y_dev, p):.4f}  [{time.time() - t0:.0f}s]")
    L.append(f"  majority-class baseline: dev macro-F1 {E.binary_report(y_dev, np.ones_like(y_dev))['macro_f1']:.4f}")

## Tables 7-8: ensembles, errors

In [4]:
# ---- Table 7: ensembles (average of P(label=1)); members chosen on dev
def avg(ts, split):
    src = DEV if split == "dev" else TEST
    return np.mean([src[t]["p1"] for t in ts], axis=0)
ranked = sorted(FOUND, key=f1, reverse=True)
tech_runs = [t for t in ranked if EXPS[t]["group"] in ("T", "X")]
best_of = lambda m: next((t for t in tech_runs if EXPS[t]["model"].upper() == m), None)
ENS = {"ens_top3": ranked[:3], "ens_top5": ranked[:5], "ens_top10": ranked[:10],
       "ens_bestpermodel": [b for b in (best_of(m) for m in MODELS) if b],
       "ens_lorma_all_models": [RUN[(m, "lorma")] for m in MODELS if (m, "lorma") in RUN],
       "ens_all_techniques": tech_runs}
ENS = {k: v for k, v in ENS.items() if len(v) >= 2}
L.append("\nTABLE 7 - ENSEMBLES: dev macro-F1 at 0.5 | AUC | dev-best threshold -> macro-F1 | members")
ENS_RES = {}
for k, ts in ENS.items():
    p = avg(ts, "dev")
    r = E.binary_report(y_dev, (p >= 0.5).astype(int)); thr, fb = E.best_threshold(y_dev, p)
    ENS_RES[k] = dict(f1=r["macro_f1"], auc=E.auc(y_dev, p), thr=thr, f1_thr=fb)
    L.append(f"  {k:22s} {r['macro_f1']:.4f} | {ENS_RES[k]['auc']:.4f} | {thr:.2f} -> {fb:.4f} | {len(ts)}: {' '.join(ts)}")

# ---- Table 8: error analysis of the best single run and the best ensemble
best_ens = max(ENS_RES, key=lambda k: ENS_RES[k]["f1"]) if ENS_RES else None
leak = re.compile(r"(appeals? (is|are|stands?) (allowed|dismissed)|(allow|dismiss) the appeal|appeal (succeeds|fails)|"
                  r"petition is (allowed|dismissed))")
n_tok = DEV[ranked[0]]["n_tok"]
terc = np.quantile(n_tok, [1 / 3, 2 / 3])
L.append("\nTABLE 8 - ERROR ANALYSIS (dev; FP = predicted 1, gold 0; FN = predicted 0, gold 1)")
for name, p in [(ranked[0], DEV[ranked[0]]["p1"])] + ([(best_ens, avg(ENS[best_ens], "dev"))] if best_ens else []):
    pr = (p >= 0.5).astype(int)
    fp, fn = (pr == 1) & (y_dev == 0), (pr == 0) & (y_dev == 1)
    L.append(f"{name}: {int(fp.sum())} FP, {int(fn.sum())} FN of {len(y_dev)}")
    for lo, hi, lab in ((0, terc[0], "short"), (terc[0], terc[1], "medium"), (terc[1], 1e12, "long")):
        s = (n_tok >= lo) & (n_tok < hi)
        L.append(f"   {lab:6s} cases (<= {min(hi, n_tok.max()):,.0f} tokens): macro-F1 {E.macro_f1(y_dev[s], pr[s]):.4f}  "
                 f"FP {int((fp & s).sum())}  FN {int((fn & s).sum())}  n {int(s.sum())}")
    has = np.array([bool(leak.search(D.clean_text(dev_docs[i]["text"])[-1500:])) for i in ref_ids])
    L.append(f"   cases whose last 1,500 characters still state the outcome ({int(has.sum())}): macro-F1 "
             f"{E.macro_f1(y_dev[has], pr[has]):.4f}; others: {E.macro_f1(y_dev[~has], pr[~has]):.4f}")
    conf = np.argsort(-np.abs(p - 0.5) * ((pr != y_dev).astype(float)))[:6]
    for j in conf:
        if pr[j] == y_dev[j]:
            continue
        L.append(f"   confident {'FP' if pr[j] else 'FN'}: {ref_ids[j]} p1={p[j]:.2f} {n_tok[j]:,} tokens | ...{D.clean_text(dev_docs[ref_ids[j]]['text'])[-220:]}")
print("\n".join(L))

=== REPORT phase 4 ===
data: {"train": {"rows_in_file": 37387, "unique_ids": 32305, "docs": 32305, "dropped_overlap_with_dev_test": 0, "tokens": 116612264, "tok_median": 2308, "tok_p90": 7371, "tok_max": 476066, "label_1": 13385, "label_0": 18920, "seconds": 184}, "dev": {"rows_in_file": 1988, "unique_ids": 994, "docs": 994, "tokens": 4372529, "tok_median": 3302, "tok_p90": 8055, "tok_max": 53526, "label_1": 497, "label_0": 497, "seconds": 8}, "test": {"rows_in_file": 1517, "unique_ids": 1517, "docs": 1517, "tokens": 6835239, "tok_median": 3441, "tok_p90": 8420, "tok_max": 50399, "label_1": 762, "label_0": 755, "seconds": 12}}
56 runs; dev 994 cases (497 label 1); test 1517 cases

ALL RUNS by dev macro-F1 (decision 0.5) | AUC | macro-F1 at the dev-best threshold | P | R | predicted-1 share | steps (best/planned) | trainable M | train min | peak GB | ms/doc
D07_full                   0.7680 | 0.8639 | 0.7877@0.45 | 0.7961 | 0.7726 |  36% | 4039/4039 |  114.99 |  70.7 |  7.2 |   35.5
D05

## Submission files

In [5]:
# ---- submission files, each a different method; dev decides the order
N_RECOMMENDED = 10                       # Codabench allows 10 per day; files beyond this are backups
picks, seen = [], {}
def add(name, p_test, thr, why, dev_f1):
    pred = (p_test >= thr).astype(int)
    key = pred.tobytes()
    if key in seen:
        print(f"skip {name}: identical to {seen[key]}"); return
    seen[key] = name
    picks.append(dict(name=name, pred=pred, thr=thr, why=why, dev=dev_f1))
if not HAVE_TEST:
    print("no test probabilities in the bundles (test.json was missing) - no submission files")
else:
    if best_ens:
        r = ENS_RES[best_ens]
        add(best_ens, avg(ENS[best_ens], "test"), 0.5, "best ensemble on dev", r["f1"])
        if r["f1_thr"] > r["f1"] + 0.005:
            add(best_ens + "_cal", avg(ENS[best_ens], "test"), r["thr"], f"same, threshold {r['thr']:.2f} chosen on dev", r["f1_thr"])
    add(ranked[0], TEST[ranked[0]]["p1"], 0.5, "best single run on dev", f1(ranked[0]))
    tech_of = lambda t: t.split("_", 1)[1]
    best_tech = {tk: next((t for t in tech_runs if tech_of(t) == tk), None) for tk in TECHS}
    if best_tech["lorma"]:                             # the compulsory technique
        add(best_tech["lorma"], TEST[best_tech["lorma"]]["p1"], 0.5, "best LoRMA run (compulsory technique)", f1(best_tech["lorma"]))
    for m in MODELS:                                   # every Phase-1 model represented (the brief asks for 4+ models)
        b = best_of(m)
        if b: add(b, TEST[b]["p1"], 0.5, f"best run of model {m}", f1(b))
    covered = {tech_of(pk["name"]) for pk in picks if pk["name"] in EXPS}
    for tk in TECHS:                                   # then every technique not yet represented
        b = best_tech[tk]
        if b and tk not in covered: add(b, TEST[b]["p1"], 0.5, f"best {tk} run", f1(b))
    for k in sorted(ENS_RES, key=lambda k: -ENS_RES[k]["f1"]):
        add(k, avg(ENS[k], "test"), 0.5, "ensemble", ENS_RES[k]["f1"])
    SUB = f"{W}/submissions"; shutil.rmtree(SUB, ignore_errors=True); os.makedirs(SUB)
    rows = []
    for n, pk in enumerate(picks, 1):
        fn = f"{n:02d}_{pk['name']}.zip"
        pj = f"{W}/_prediction.json"
        D.write_prediction(test_ids, pk["pred"], pj)
        with zipfile.ZipFile(f"{SUB}/{fn}", "w", zipfile.ZIP_DEFLATED) as z:
            z.write(pj, arcname="prediction.json")
        rows.append(f"{fn},{pk['dev']:.4f},{pk['thr']:.2f},{int(pk['pred'].sum())},{len(test_ids)},\"{pk['why']}\"")
    open(f"{W}/SUBMISSIONS.csv", "w").write("file,dev_macro_f1,threshold,test_pred_1,test_cases,why\n" + "\n".join(rows) + "\n")
    with zipfile.ZipFile(f"{W}/all_submissions.zip", "w") as z:
        for f in sorted(os.listdir(SUB)):
            z.write(f"{SUB}/{f}", arcname=f)
        z.write(f"{W}/SUBMISSIONS.csv", arcname="SUBMISSIONS.csv")
    L.append(f"\nSUBMISSIONS (submit in this order; the first {N_RECOMMENDED} are recommended, the rest are backups)")
    L += ["  " + r for r in rows]
    print("\n".join(L[-len(rows) - 1:]))
open(f"{W}/REPORT_p4_final.txt", "w").write("\n".join(L))
print("\n=== END REPORT ===")

skip D07_full: identical to D07_full
skip ens_top5: identical to ens_top5

SUBMISSIONS (submit in this order; the first 10 are recommended, the rest are backups)
  01_ens_top5.zip,0.7835,0.50,589,1517,"best ensemble on dev"
  02_ens_top5_cal.zip,0.7954,0.42,702,1517,"same, threshold 0.42 chosen on dev"
  03_D07_full.zip,0.7680,0.50,576,1517,"best single run on dev"
  04_C01_lorma.zip,0.7404,0.50,589,1517,"best LoRMA run (compulsory technique)"
  05_A06_partial.zip,0.7146,0.50,627,1517,"best run of model A"
  06_B06_partial.zip,0.7515,0.50,597,1517,"best run of model B"
  07_C07_full.zip,0.7559,0.50,568,1517,"best run of model C"
  08_E05_gradual.zip,0.7256,0.50,530,1517,"best run of model E"
  09_D02_adalora.zip,0.7380,0.50,595,1517,"best adalora run"
  10_E03_adapter.zip,0.7162,0.50,504,1517,"best adapter run"
  11_B04_bitfit.zip,0.6091,0.50,836,1517,"best bitfit run"
  12_D08_lora.zip,0.7523,0.50,601,1517,"best lora run"
  13_ens_top3.zip,0.7832,0.50,590,1517,"ensemble"
  14_ens_best